In [0]:
# importar bibliotecas
import re
import unicodedata
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.functions import col

In [0]:
#👉 Função para padronizar nomes de colunas

def clean_column_name(col_name):
    col_name = unicodedata.normalize('NFKD', col_name)
    col_name = col_name.encode('ascii', 'ignore').decode('utf-8')
    col_name = col_name.lower()
    col_name = col_name.replace(" ", "_")
    col_name = re.sub(r'[^a-z0-9_]', '', col_name)
    col_name = re.sub(r'_+$', '', col_name)   # ← remove _ somente no final
    return col_name

In [0]:
# Elimina duplicados após a limpeza

def make_unique_columns(columns):
    counts = {}
    new_cols = []

    for c in columns:
        clean_c = clean_column_name(c)
        if clean_c in counts:
            counts[clean_c] += 1
            clean_c = f"{clean_c}_{counts[clean_c]}"
        else:
            counts[clean_c] = 0
        new_cols.append(clean_c)

    return new_cols

In [0]:
# análise exploratória
# Lê o arquivo CSV

caminho_scv = "/Volumes/workspace/pipeline_estudo/raw_files/censo_padre_bernardo/local_votacao_secao.csv"
df = spark.read\
    .format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .option("sep", ";")\
    .option("multiLine", True)\
    .option("quote", '"')\
    .load(caminho_scv)
display(df)

In [0]:
# Definir caminhos

source_path = "/Volumes/workspace/pipeline_estudo/raw_files/censo_padre_bernardo/"
checkpoint_path = "/Volumes/workspace/pipeline_estudo/raw_files/checkpoints/Local_votacao_secao/bronze" 
# salva o "ponteiro" de onde o Spark parou. Se você rodar o código hoje e amanhã, ele só processará os arquivos que surgiram nesse intervalo.


In [0]:
from pyspark.sql.functions import col
# Lê os arquivos com Autoloader

# Schema com as mesmas opções do arquivo real
name_file = "local_votacao_secao.csv"
schema = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ";")
    .option("multiLine", True)
    .option("quote", '"')
    .csv(source_path + name_file)  # arquivo específico
    .schema
)

# Auto Loader
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("pathGlobFilter", name_file)
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("sep", ";")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .schema(schema)
    .load(source_path)
)

# Aplica padronização  das colunas
df_padronizado = (df.toDF(*make_unique_columns(df.columns))
.withColumn("zona", lit(131)) # Adiciona a coluna fixa
)

# Preview estático — mesmo arquivo, mesmas opções
df_preview = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("sep", ";")
    .csv(source_path + name_file)
)

df_padronizado_preview = (df_preview.toDF(*make_unique_columns(df_preview.columns))
.withColumn("zona", lit(131))) # Adiciona a coluna fixa

display(df_padronizado_preview, checkpointLocation = checkpoint_path)

In [0]:
# limpa caracteres especiais

from pyspark.sql.functions import regexp_replace, trim, lower
from pyspark.sql import functions as F

def fix_encoding(df):
    string_cols = [f.name for f in df.schema.fields if str(f.dataType) == "StringType()"]
    for c in string_cols:
        df = df.withColumn(c,
            regexp_replace(
                F.decode(F.encode(F.col(c), "utf-8"), "utf-8"),
                r'[^\x20-\x7EÀ-ÿ]', ''   # mantém ASCII + Latin (acentos normais)
            )
        )
        df = df.withColumn(c, lower(trim(df[c])))   # ← lower() junto com trim()
    return df

# Aplica nos dois DataFrames
df_padronizado        = fix_encoding(df_padronizado)
df_padronizado_preview = fix_encoding(df_padronizado_preview)


In [0]:
# Adiciona colunas técnicas da Bronze — derivado do STREAM
df_bronze = (
    df_padronizado                                          # ← stream, não df_filtrado
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("Local_votacao_secao.xlsx"))
)

# Preview com filtro — derivado do ESTÁTICO, só para display()
df_bronze_preview = (
    df_padronizado_preview
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("Local_votacao_secao.xlsx"))
)

In [0]:
# Reseta o checkpoint para reprocessar do zero
dbutils.fs.mkdirs(checkpoint_path)
print("Checkpoint OK.")

In [0]:
# Limpa tabela e checkpoint para regravar com filtro
dbutils.fs.rm(checkpoint_path, recurse=True)
dbutils.fs.mkdirs(checkpoint_path)

spark.sql("DROP TABLE IF EXISTS workspace.drs_bronze.Local_votacao_secao")

print("Tabela e checkpoint limpos. Rode a célula de writeStream.")

In [0]:
# Grava o stream na tabela Bronze final
(df_bronze.writeStream                          # ← df_bronze + writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("workspace.drs_bronze.local_votacao_secao")
)

In [0]:
# Valida a tabela Bronze final
spark.sql("""
SELECT *
FROM workspace.drs_bronze.local_votacao_secao
LIMIT 10
""").display()